In [ ]:
# Example 1: Top-level public API surface.
import langchain_xai

public_names = [n for n in dir(langchain_xai) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(langchain_xai, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, "Expected at least one public name in langchain_xai"


In [ ]:
# Example 2: Import the ChatXAI class — the primary entrypoint.
from langchain_xai import ChatXAI

print(f"ChatXAI: {ChatXAI}")
print(f"ChatXAI module: {ChatXAI.__module__}")
print(f"ChatXAI MRO: {[c.__name__ for c in ChatXAI.__mro__[:5]]}")

# ChatXAI should be a Pydantic-style model (inherits from BaseChatModel)
assert hasattr(ChatXAI, 'model_fields') or hasattr(ChatXAI, '__fields__'), \
    "ChatXAI should expose Pydantic model fields"


In [ ]:
# Example 3: Instantiate ChatXAI without making any network calls.
import os
from langchain_xai import ChatXAI

# Use a dummy API key to avoid AuthN; we only check construction here.
os.environ.setdefault('XAI_API_KEY', 'sk-test-dummy-not-real')

chat = ChatXAI(model='grok-2-latest', temperature=0.0)
print(f"chat: {chat!r}"[:200])
print(f"chat.model_name or model: {getattr(chat, 'model_name', getattr(chat, 'model', None))}")
print(f"chat.temperature: {chat.temperature}")

assert chat.temperature == 0.0


In [ ]:
# Example 4: ChatXAI inherits from langchain_core BaseChatModel.
from langchain_xai import ChatXAI
from langchain_core.language_models.chat_models import BaseChatModel

assert issubclass(ChatXAI, BaseChatModel), \
    "ChatXAI should be a subclass of langchain_core BaseChatModel"

# Core BaseChatModel methods should all be present.
expected_methods = ['invoke', 'ainvoke', 'stream', 'astream', 'batch']
for m in expected_methods:
    assert hasattr(ChatXAI, m), f"ChatXAI missing expected method {m!r}"
    print(f"  {m}: {getattr(ChatXAI, m)}")

print("\nChatXAI conforms to BaseChatModel interface")


In [ ]:
# Example 5: Bind tools / structured output API is present.
import os
from langchain_xai import ChatXAI

os.environ.setdefault('XAI_API_KEY', 'sk-test-dummy-not-real')
chat = ChatXAI(model='grok-2-latest')

# Both bind_tools and with_structured_output are common LangChain extension points.
assert hasattr(chat, 'bind_tools'), "ChatXAI should support bind_tools"
assert hasattr(chat, 'with_structured_output'), "ChatXAI should support with_structured_output"

# Bind a no-op tool just to confirm the wiring doesn't raise.
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"sunny in {city}"

bound = chat.bind_tools([get_weather])
print(f"bound: {type(bound).__name__}")
assert bound is not None
print("ChatXAI tool-binding wiring works")
